In [7]:
import geopandas as gpd
from pathlib import Path
import ee
import os
import pandas as pd
import geopandas as gpd
import json
import datetime



In [2]:
import importlib
import sys

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import label_utils
importlib.reload(label_utils)

from label_utils import (
    get_s2_s1_matching_dates,
    get_s2_all_dates_with_s1,
    image_details_to_json,
    plot_dates,
    select_comparison_dates,
)

In [4]:


from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

True

In [5]:
ee.Initialize(project=os.environ["EE_PROJECT"])

In [6]:
site_fp = PROJECT_ROOT / "configs" / "sites.json"
images_fp = PROJECT_ROOT / "configs"/ "images.json"


In [8]:
ee_list_matching_dates = get_s2_s1_matching_dates(location = "RawaPening",project_root= PROJECT_ROOT, site_fp= site_fp, cloud_perc= 50)

In [15]:
print(type(ee_list_matching_dates))
print(ee_list_matching_dates["date"].unique())

<class 'pandas.core.frame.DataFrame'>
['2019-01-09' '2019-03-10' '2019-03-30' '2019-04-09' '2019-05-09'
 '2019-05-29' '2019-06-08' '2019-07-08' '2019-07-28' '2019-09-06'
 '2019-09-26' '2019-10-06' '2019-11-05' '2019-11-25' '2020-01-24'
 '2020-02-03' '2020-04-03' '2020-07-02' '2020-08-01' '2020-08-31'
 '2020-09-20' '2020-09-30' '2020-11-19' '2021-02-27' '2021-03-19'
 '2021-05-28' '2021-07-27' '2021-08-26' '2021-09-25' '2021-10-25'
 '2021-12-24' '2022-03-14' '2023-03-09' '2023-06-17' '2023-08-16'
 '2023-09-05' '2023-10-15' '2023-12-14' '2024-02-12' '2024-04-12'
 '2024-05-02' '2024-06-11' '2024-07-01' '2024-08-10' '2024-08-30'
 '2024-10-29' '2025-02-06' '2025-04-07' '2025-05-07' '2025-05-19'
 '2025-06-06' '2025-06-26' '2025-07-18' '2025-08-25' '2025-09-04'
 '2025-10-16' '2025-12-23']


In [11]:
# Import vembanad matching dates 
dates_fp = PROJECT_ROOT /"outputs/RawaPening_matching_dates.csv"
matching_dates_df = pd.read_csv(dates_fp)

In [12]:
fig = plot_dates(location = "RawaPening", project_root= PROJECT_ROOT, start_year= 2022, end_year=2025)
fig.show()

In [28]:
# Next want a function that takes a matching dates Df and writes the info into images.json

# How to chose the comparison images???

comparison_df = select_comparison_dates(matching_dates= matching_dates_df, months_back= 3, max_cloud_perc= 60)

In [51]:
comparison_df.head()

,Unnamed: 0,S1_img_id,S1_time,S2_img_id,S2_time,cloud_perc,date,location,time_diff,season,...,comparison_1_date,comparison_1_s2_img_id,comparison_1_s1_img_id,comparison_1_cloud_perc,comparison_1_age_days,comparison_2_date,comparison_2_s2_img_id,comparison_2_s1_img_id,comparison_2_cloud_perc,comparison_2_age_days
0,1,COPERNICUS/S1_GRD/S1B_IW_GRDH_1SDV_20190107T00...,2019-01-07 00:39:13+00:00,COPERNICUS/S2_SR_HARMONIZED/20190107T170701_20...,2019-01-07 17:18:20.897000+00:00,0.012322,2019-01-07,Valsequillo,0 days 16:39:07.897000,winter,...,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
1,3,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20190122T12...,2019-01-22 12:26:09+00:00,COPERNICUS/S2_SR_HARMONIZED/20190122T170619_20...,2019-01-22 17:18:26.325000+00:00,0.000185,2019-01-22,Valsequillo,0 days 04:52:17.325000,winter,...,2019-01-07,COPERNICUS/S2_SR_HARMONIZED/20190107T170701_20...,COPERNICUS/S1_GRD/S1B_IW_GRDH_1SDV_20190107T00...,0.012322,15.0,NaT,NaN,NaN,NaN,NaN
2,4,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20190206T00...,2019-02-06 00:40:10+00:00,COPERNICUS/S2_SR_HARMONIZED/20190206T170501_20...,2019-02-06 17:18:37.029000+00:00,0.000266,2019-02-06,Valsequillo,0 days 16:38:27.029000,winter,...,2019-01-22,COPERNICUS/S2_SR_HARMONIZED/20190122T170619_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20190122T12...,0.000185,15.0,2019-01-07,COPERNICUS/S2_SR_HARMONIZED/20190107T170701_20...,COPERNICUS/S1_GRD/S1B_IW_GRDH_1SDV_20190107T00...,0.012322,30.0
3,6,COPERNICUS/S1_GRD/S1B_IW_GRDH_1SDV_20190221T12...,2019-02-21 12:25:14+00:00,COPERNICUS/S2_SR_HARMONIZED/20190221T170329_20...,2019-02-21 17:18:39.407000+00:00,0.006086,2019-02-21,Valsequillo,0 days 04:53:25.407000,winter,...,2019-01-22,COPERNICUS/S2_SR_HARMONIZED/20190122T170619_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20190122T12...,0.000185,30.0,2019-02-06,COPERNICUS/S2_SR_HARMONIZED/20190206T170501_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20190206T00...,0.000266,15.0
4,8,COPERNICUS/S1_GRD/S1B_IW_GRDH_1SDV_20190308T00...,2019-03-08 00:39:11+00:00,COPERNICUS/S2_SR_HARMONIZED/20190308T170131_20...,2019-03-08 17:18:34.608000+00:00,0.000448,2019-03-08,Valsequillo,0 days 16:39:23.608000,spring,...,2019-01-22,COPERNICUS/S2_SR_HARMONIZED/20190122T170619_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20190122T12...,0.000185,45.0,2019-01-07,COPERNICUS/S2_SR_HARMONIZED/20190107T170701_20...,COPERNICUS/S1_GRD/S1B_IW_GRDH_1SDV_20190107T00...,0.012322,60.0


In [55]:
# Write image details to images.json from camparison_df that is the output from select_comparison_dates()

image_dict = image_details_to_json(images_fp= images_fp, comparison_df= comparison_df)

NameError: name 'image_details_to_json' is not defined

In [10]:
# Loop through rest of locations:

with open(site_fp) as f:
    sites = json.load(f)

location_list = list(sites.get("sites", {}).keys())

for location in location_list:
    matching_dates = get_s2_s1_matching_dates(location, project_root= PROJECT_ROOT, site_fp= site_fp, cloud_perc= 40)

    comparison_df = select_comparison_dates(matching_dates= matching_dates, months_back= 4, n_comparisons=2, max_cloud_perc=40, min_gap_days=7)

    fig= plot_dates(location=location, start_year= 2021, end_year=2025, project_root= PROJECT_ROOT)
    fig.show()
    image_dict = image_details_to_json(images_fp= images_fp, comparison_df= comparison_df)
    print(f"Ran for {location}")

Ran for Vembanad


Ran for Winam


Ran for Inle


Ran for Hartbeespoort


Ran for Mula


Ran for RawaPening


Ran for Rodman


Ran for Valsequillo


In [21]:
#### To get more images for locations with few matching overpasses run an different function that does a left join of s1 onto s2 instead of the inner join
# so can see more image choices even if no match. Also useful for extra comparison images. 

location_subset = ["RawaPening", "Rodman", "Mula", "Vembanad"]

for location in location_subset:
    dates = get_s2_all_dates_with_s1(site_fp=site_fp, location= location, project_root= PROJECT_ROOT, cloud_perc= 30)

    comparison_df = select_comparison_dates(matching_dates= dates, months_back= 6, n_comparisons=2, max_cloud_perc=10, min_gap_days= 14)

    image_dict = image_details_to_json(images_fp= images_fp, comparison_df= comparison_df)
    print(f"Ran for {location}")

    

Ran for RawaPening
Ran for Rodman
Ran for Mula
Ran for Vembanad


### Create table of number of labels


In [5]:
samples = PROJECT_ROOT / "outputs" / "sample_points_clean.csv"

cleaned_samples = pd.read_csv(samples)

cleaned_samples.head()

,system:index,AWEIp95,B11,B12,B2,B3,B4,B5,B6,B7,...,B8A,class_int,class_label,label_id,latitude,lc,location,longitude,obs_date,.geo
0,0_0_0,2161,294,174,431,584,327,444,776,759,...,799,3,floating_plants,VE0001_20221224,9.689175,1,Vembanad,76.393321,2022-12-24,"{""type"":""MultiPoint"",""coordinates"":[]}"
1,0_1_0,2191,256,133,406,472,301,367,564,553,...,557,3,floating_plants,VE0002_20221224,9.689048,1,Vembanad,76.393986,2022-12-24,"{""type"":""MultiPoint"",""coordinates"":[]}"
2,0_2_0,2214,173,112,430,506,323,382,479,477,...,500,3,floating_plants,VE0003_20221224,9.689937,1,Vembanad,76.397332,2022-12-24,"{""type"":""MultiPoint"",""coordinates"":[]}"
3,0_3_0,2239,254,119,431,478,301,429,683,684,...,753,3,floating_plants,VE0004_20221224,9.689429,1,Vembanad,76.399456,2022-12-24,"{""type"":""MultiPoint"",""coordinates"":[]}"
4,0_4_0,2148,144,93,464,506,330,372,430,415,...,405,3,floating_plants,VE0005_20221224,9.689323,1,Vembanad,76.398770,2022-12-24,"{""type"":""MultiPoint"",""coordinates"":[]}"


In [ ]:
cleaned_samples["obs_date"] = pd.to_datetime(cleaned_samples["obs_date"],format="%Y-%m-%d")
cleaned_samples = cleaned_samples[["class_label", "lc", "class_int", "location", "obs_date"]]

grouped_samples = cleaned_samples.drop(columns =["class_int"]).groupby(["location", "lc", "class_label"]).count()

In [27]:
grouped_samples

obs_date
location      lc class_label              
Hartbeespoort 0  LEV                   509
                 open_water            415
                 surface_algae          81
              1  floating_plants      1002
Inle          0  LEV                   500
                 open_water            480
              1  floating_plants       999
Mula          0  LEV                   576
                 open_water            493
              1  floating_plants      1003
RawaPening    0  LEV                   510
                 open_water            490
              1  floating_plants      1002
Rodman        0  LEV                   521
                 open_water            370
              1  floating_plants      1021
Valsequillo   0  LEV                   477
                 open_water            428
                 surface_algae          33
              1  floating_plants       955
Vembanad      0  LEV                   500
                 open_water            500
              1  floating_plants      1002
Winam         0  LEV                   289
                 open_water            280
                 surface_algae         430
              1  floating_plants      1000

In [57]:
season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn",
}

# Can make seson an ordered categorical so it list in season order rather than alphabetical
from pandas.api.types import CategoricalDtype


by_date = cleaned_samples.copy()
by_date["year"] = by_date["obs_date"].dt.year
by_date["month"] = by_date["obs_date"].dt.month




season_order = CategoricalDtype(
    ["Winter", "Spring", "Summer", "Autumn"],
    ordered=True,
)

by_date["season"] = (
    by_date["obs_date"]
    .dt.month
    .map(season_map)
    .astype(season_order)
)

by_date = by_date.drop(columns=["class_int"]).groupby(["location", "year", "season"]).count()
by_date = by_date.loc[by_date["obs_date"]!= 0]

/var/folders/px/ysdd4tvd6z7255np16x_10l80000gn/T/ipykernel_52694/1000513744.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  by_date = by_date.drop(columns=["class_int"]).groupby(["location", "year", "season"]).count()


In [60]:
by_date.head(20)

class_label   lc  obs_date  month
location      year season                                   
Hartbeespoort 2021 Winter          100  100       100    100
                   Spring          107  107       107    107
                   Summer          190  190       190    190
                   Autumn          200  200       200    200
              2022 Spring          190  190       190    190
                   Summer          200  200       200    200
              2023 Winter          300  300       300    300
              2024 Winter          200  200       200    200
                   Summer          225  225       225    225
                   Autumn          120  120       120    120
              2025 Summer          175  175       175    175
Inle          2019 Winter           70   70        70     70
                   Autumn          500  500       500    500
              2020 Winter          229  229       229    229
                   Autumn          470  470       470    470
              2021 Winter           80   80        80     80
                   Autumn          260  260       260    260
              2023 Winter           97   97        97     97
              2024 Winter          140  140       140    140
              2025 Winter           94   94        94     94